In [1]:
%pip install h5py scikit-learn

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ------------------------- -------------- 1.8/2.9 MB 25.4 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 12.9 MB/s  0:00:00
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   -------------- ------------------------- 2.9/8.0 MB 15.2 MB/s eta 0:00:01
   ---------------------------- ----------- 5.8/8.0 MB 14.1 MB/s eta 0:00:01
   ---------------------------------------  7.9/8.0 MB 13.5 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 12.7 MB/s  0:00:00

   ------------- -------------------------- 1/3 [h5py]
   ------------- -------------------------- 1/3 [h5py]
   ------------- -------------------------- 1/3 [h5py]
   ------------- -------------------------- 1/3 [h5py]
   ------------- -------------------------- 1/3 [h5py]
   ------------- -------------------------- 1/3 [h5p

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import h5py
import numpy as np

# Nombres reales de las 10 clases de galaxias en el dataset DECals
CLASES_GALAXY10 = [
    "Disco, De frente, Sin Espiral",
    "Suave, Completamente redonda (Elíptica)",
    "Suave, Intermedia",
    "Suave, Forma de cigarro",
    "Disco, De canto, Bulbo redondo",
    "Disco, De canto, Bulbo cuadrado",
    "Disco, De canto, Sin bulbo",
    "Disco, De frente, Espiral apretada",
    "Disco, De frente, Espiral media",
    "Disco, De frente, Espiral suelta"
]

# =================================================================
# 1. LECTOR PERSONALIZADO DE HDF5 (.h5)
# =================================================================
class LectorGalaxy10(Dataset):
    def __init__(self, imagenes, etiquetas, transform=None):
        self.imagenes = imagenes
        self.etiquetas = etiquetas
        self.transform = transform

    def __len__(self):
        return len(self.etiquetas)

    def __getitem__(self, idx):
        # La imagen viene como un arreglo de Numpy (256, 256, 3)
        img = self.imagenes[idx]
        etiqueta = self.etiquetas[idx]
        
        if self.transform:
            img = self.transform(img)
            
        return img, etiqueta

# =================================================================
# 2. LA RED NEURONAL CONVOLUCIONAL (Adaptada a Color y 10 clases)
# =================================================================
class AstroNetDECals(nn.Module):
    def __init__(self):
        super(AstroNetDECals, self).__init__()
        self.conv_layers = nn.Sequential(
            # CAMBIO: 3 canales de entrada (RGB) en lugar de 1
            nn.Conv2d(3, 16, kernel_size=3, padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128), # Asumiendo que reduciremos las imágenes a 64x64
            nn.ReLU(),
            nn.Dropout(0.5),
            # CAMBIO: 10 salidas porque hay 10 tipos de galaxias
            nn.Linear(128, 10) 
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# =================================================================
# 3. ENTRENAMIENTO MASIVO
# =================================================================
def entrenar_con_h5(ruta_h5, epocas=40):
    print("🔭 Extrayendo el universo desde el archivo HDF5...")
    
    # 1. Extraer los datos a la memoria RAM (Puede tomar unos segundos)
    with h5py.File(ruta_h5, 'r') as F:
        imagenes = np.array(F['images'])
        etiquetas = np.array(F['ans'])
        
    print(f"✅ ¡Extraídas {len(imagenes)} galaxias!")
    
    # 2. Dividir: 80% para entrenar, 20% para examinar a la IA
    X_train, X_val, y_train, y_val = train_test_split(imagenes, etiquetas, test_size=0.2, random_state=42)

    # 3. Transformaciones con "Aumento de Datos" (El truco maestro)
    transformaciones_train = transforms.Compose([
        transforms.ToPILImage(), # Necesario para rotar el arreglo
        transforms.RandomHorizontalFlip(p=0.5), # Voltea la galaxia como espejo el 50% de las veces
        transforms.RandomVerticalFlip(p=0.5),   # La voltea de cabeza
        transforms.RandomRotation(degrees=45),  # La gira al azar hasta 45 grados
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    # OJO: La validación NO se rota (queremos examinar a la IA con fotos normales)
    transformaciones_val = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    # 4. Crear los Cargadores con sus respectivas transformaciones
    train_dataset = LectorGalaxy10(X_train, y_train, transform=transformaciones_train)
    val_dataset = LectorGalaxy10(X_val, y_val, transform=transformaciones_val)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"⚡ Entrenando en: {device}")

    modelo = AstroNetDECals().to(device)
    criterio = nn.CrossEntropyLoss()
    optimizador = optim.Adam(modelo.parameters(), lr=0.001)

    print("\n🚀 Iniciando entrenamiento morfológico...")
    for epoca in range(epocas):
        modelo.train()
        perdida_total = 0.0
        
        for imgs, etqs in train_loader:
            # Los datos en el HDF5 suelen venir como uint8, ToTensor los pasa a float automáticamente
            imgs = imgs.to(device)
            # PyTorch requiere que las etiquetas de clase sean 'long' (enteros de 64 bits)
            etqs = etqs.to(device, dtype=torch.long) 
            
            optimizador.zero_grad()
            predicciones = modelo(imgs)
            perdida = criterio(predicciones, etqs)
            perdida.backward()
            optimizador.step()
            perdida_total += perdida.item()

        # Validación
        modelo.eval()
        correctos = 0
        total = 0
        with torch.no_grad():
            for imgs, etqs in val_loader:
                imgs = imgs.to(device)
                etqs = etqs.to(device, dtype=torch.long)
                predicciones = modelo(imgs)
                _, adivinanza = torch.max(predicciones.data, 1)
                total += etqs.size(0)
                correctos += (adivinanza == etqs).sum().item()

        precision = 100 * correctos / total
        print(f"Época [{epoca+1}/{epocas}] | Error: {perdida_total/len(train_loader):.4f} | Precisión Validación: {precision:.2f}%")

    torch.save(modelo.state_dict(), "clasificador_galaxias_decals.pt")
    print("\n💾 ¡Modelo guardado como 'clasificador_galaxias_decals.pt'!")

# =================================================================
# MAIN
# =================================================================
if __name__ == '__main__':
    # Pon aquí la ruta de tu archivo Galaxy10_DECals.h5
    archivo_h5 = "Galaxy10_DECals.h5"
    
    # Descomenta la línea de abajo para iniciar el entrenamiento
    entrenar_con_h5(archivo_h5, epocas=40)

🔭 Extrayendo el universo desde el archivo HDF5...
✅ ¡Extraídas 17736 galaxias!
⚡ Entrenando en: cuda

🚀 Iniciando entrenamiento morfológico...
Época [1/40] | Error: 1.9579 | Precisión Validación: 43.29%
Época [2/40] | Error: 1.5515 | Precisión Validación: 49.80%
Época [3/40] | Error: 1.4131 | Precisión Validación: 57.92%
Época [4/40] | Error: 1.3057 | Precisión Validación: 60.79%
Época [5/40] | Error: 1.2464 | Precisión Validación: 61.53%
Época [6/40] | Error: 1.1985 | Precisión Validación: 63.39%
Época [7/40] | Error: 1.1493 | Precisión Validación: 65.16%
Época [8/40] | Error: 1.1253 | Precisión Validación: 65.81%
Época [9/40] | Error: 1.1009 | Precisión Validación: 67.42%
Época [10/40] | Error: 1.0790 | Precisión Validación: 67.31%
Época [11/40] | Error: 1.0718 | Precisión Validación: 68.12%
Época [12/40] | Error: 1.0449 | Precisión Validación: 69.42%
Época [13/40] | Error: 1.0285 | Precisión Validación: 69.81%
Época [14/40] | Error: 1.0108 | Precisión Validación: 68.49%
Época [15/40